<a href="https://colab.research.google.com/github/munozsan03/Astro_Team4/blob/feature%2Fdata-pipeline/Astro_Team4_Data_Pipeline_Torchlight_Hackathon_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Welcome to the 2026 Torchlight Hackathon!

Your challenge for this hackathon is to help astronauts better understand the effects their space travel has had on their overall wellbeing. You'll use the individual astronaut data from the [2021 Inspiration 4 mission](https://inspiration4.com/).

In this notebook, you can access, read in, and download data files from the Inspiration 4 astronauts and their spacecraft, using the [Developer API](https://www.nasa.gov/reference/osdr-developer-api/) from the [NASA Open Science Data Repository](https://science.nasa.gov/biological-physical/data/osdr/).

In [9]:
import os, re
import pandas as pd

# ── Timepoint normalisation ───────────────────────────────────────────────────
# Maps every messy variant found across the 19 datasets → (canonical, phase)
_TP_MAP = {
    "l-92":"L-92","l_92":"L-92","l92":"L-92",
    "l-44":"L-44","l_44":"L-44","l44":"L-44",
    "l-3":"L-3","l_3":"L-3","l3":"L-3",
    "fd2":"FD2","fd_2":"FD2","fd 2":"FD2","day2":"FD2",
    "fd3":"FD3","fd_3":"FD3","fd 3":"FD3","day3":"FD3",
    "r+1":"R+1","r_1":"R+1","r1":"R+1",
    "r+45":"R+45","r_45":"R+45","r45":"R+45",
    "r+82":"R+82","r_82":"R+82","r82":"R+82",
}
_PHASE_MAP = {
    "L-92":"Preflight","L-44":"Preflight","L-3":"Preflight",
    "FD2":"Inflight","FD3":"Inflight",
    "R+1":"Postflight","R+45":"Postflight","R+82":"Postflight",
}
CANONICAL_TIMEPOINTS = ["L-92","L-44","L-3","FD2","FD3","R+1","R+45","R+82"]

def standardize_timepoint(val):
    """Return canonical timepoint string, or 'UNKNOWN'."""
    s = str(val).lower().strip()
    if s in _TP_MAP:
        return _TP_MAP[s]
    for k, v in _TP_MAP.items():
        if k in s:
            return v
    return "UNKNOWN"

def get_phase(tp):
    return _PHASE_MAP.get(tp, "UNKNOWN")

# ── Crew ID normalisation ─────────────────────────────────────────────────────
# Handles: C001, CM02, crewmember3, crewID4, SUBJECT_ID_01, SUB02, S3, M4 …
_CREW_RE = re.compile(
    r"(?:C(?:M|rew(?:member|ID)?|rw)?[-_]?0*([1-4])"
    r"|(?:SUBJECT(?:_ID)?|SUB|MEMBER)[-_]?0*([1-4])"
    r"|(?:S|M)[-_]?0*([1-4])(?!\d))",
    re.IGNORECASE,
)

def standardize_crew_id(val):
    """Return canonical crew ID (C001–C004), or 'UNKNOWN'."""
    m = _CREW_RE.search(str(val))
    if m:
        digit = next(g for g in m.groups() if g is not None)
        return f"C{int(digit):03d}"
    return "UNKNOWN"

# ── Column-header parser (wide metagenomics tables) ───────────────────────────
_TP_COL_RE = re.compile(r"(L[-_]92|L[-_]44|L[-_]3|FD[-_]?[23]|R[+_]\d+)", re.IGNORECASE)

def parse_col_meta(col):
    """Extract (crew_id, timepoint, phase) from a column name string."""
    crew = standardize_crew_id(col)
    m = _TP_COL_RE.search(str(col))
    tp   = standardize_timepoint(m.group(1)) if m else "UNKNOWN"
    return crew, tp, get_phase(tp)

def add_col_metadata(df):
    """
    For wide tables (rows=features, cols=samples): attach a .attrs['meta']
    DataFrame with crew_id / timepoint / phase for every sample column.
    """
    rows = []
    for col in df.columns:
        crew, tp, phase = parse_col_meta(str(col))
        rows.append({"sample_col": col, "crew_id": crew,
                     "timepoint": tp, "phase": phase})
    df.attrs["meta"] = pd.DataFrame(rows).set_index("sample_col")
    return df

def tag_long(df, crew_col=None, timepoint_col=None):
    """
    For long tables (rows=samples): normalise crew_id / timepoint / phase
    columns in-place.
    crew_col      – existing column whose values contain the raw crew ID
    timepoint_col – existing column whose values contain the raw timepoint
    """
    df = df.copy()
    if crew_col and crew_col in df.columns:
        df["crew_id"] = df[crew_col].apply(standardize_crew_id)
    elif "crew_id" not in df.columns:
        # Try to extract from the DataFrame index
        df["crew_id"] = df.index.to_series().apply(standardize_crew_id)

    if timepoint_col and timepoint_col in df.columns:
        df["timepoint"] = df[timepoint_col].apply(standardize_timepoint)
    elif "timepoint" not in df.columns:
        df["timepoint"] = df.index.to_series().apply(standardize_timepoint)

    df["phase"] = df["timepoint"].apply(get_phase)
    return df

print("✅ Standardisation helpers loaded.")
print(f"   Canonical timepoints: {CANONICAL_TIMEPOINTS}")


✅ Standardisation helpers loaded.
   Canonical timepoints: ['L-92', 'L-44', 'L-3', 'FD2', 'FD3', 'R+1', 'R+45', 'R+82']


## 1. Oral, Nasal, and Skin Metagenomic Microbial Swabs

Dataset ID: [OSD-572](https://osdr.nasa.gov/bio/repo/data/studies/OSD-572)

The crew collected biospecimen samples before, during, and after flight. One of these biospecimen collections included oral, nasal, and skin microbial swabs collected from ten locations (oral, nasal cavity, post-auricular, axillary vault, volar forearm, occiput, umbilicus, gluteal crease, glabella, and toe web space). Swabs were collected at three pre-flight timespoints (L-92, L-44, L-3), two in-flight timepoints (flight day 2 (FD2), and flight day 3 (FD3)), and three post-flight timepoints (R+1, R+45, R+82).

There are several different ways to quantify how the microbial populations are changing in each swab at each timepoint. Below we read in 4 different files which quantify 4 different aspects of the microbial population in each swab:

* microbial function using [KEGG orthology](https://www.genome.jp/kegg/ko.html)
* microbial taxon abundance
* molecular pathway activity
* gene family abundance


More information on the metagenomics bioinformatics pipeline that generated these files is available [here](https://github.com/nasa/GeneLab_Data_Processing/tree/master/Metagenomics/Illumina).

### KEGG Orthology (KO) Metagenomics

Each column is a microbial swab sample taken from a crew member either during or after flight, from a specific anatomical region.

Each row is a [KEGG orthology](https://www.genome.jp/kegg/ko.html) molecular function that has been quantified within that sample.

In [10]:
kegg_metagenomics_crew = pd.read_csv(
    'https://osdr.nasa.gov/geode-py/ws/studies/OSD-572/download?source=datamanager'
    '&file=GLDS-564_GMetagenomics_Combined-gene-level-KO-function-coverages_GLmetagenomics.tsv',
    index_col=0, sep='\t')
kegg_metagenomics_crew = add_col_metadata(kegg_metagenomics_crew)
kegg_metagenomics_crew.head()

ValueError: The truth value of a DataFrame is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

ValueError: The truth value of a DataFrame is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

### Taxonomy Metagenomics

Each column is a microbial swab sample taken from a crew member either during or after flight, from a specific anatomical region.

Each row is a different microbial taxon that has been quantified within that sample.

In [11]:
tax_metagenomics_crew = pd.read_csv(
    'https://osdr.nasa.gov/geode-py/ws/studies/OSD-572/download?source=datamanager'
    '&file=GLDS-564_GMetagenomics_Combined-gene-level-taxonomy-coverages-CPM_GLmetagenomics.tsv',
    index_col=0, sep='\t')
tax_metagenomics_crew = add_col_metadata(tax_metagenomics_crew)
tax_metagenomics_crew.head()

/tmp/ipykernel_57369/3071231737.py:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  tax_metagenomics_crew = pd.read_csv(


ValueError: The truth value of a DataFrame is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

ValueError: The truth value of a DataFrame is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

### Pathway Metagenomics

Each column is a microbial swab sample taken from a crew member either during or after flight, from a specific anatomical region.

Each row is a different molecular pathway whose activity was quantified within that sample.

In [12]:
pathway_metagenomics_crew = pd.read_csv(
    'https://osdr.nasa.gov/geode-py/ws/studies/OSD-572/download?source=datamanager'
    '&file=GLDS-564_GMetagenomics_Pathway-abundances-cpm_GLmetagenomics.tsv',
    index_col=0, sep='\t')
pathway_metagenomics_crew = add_col_metadata(pathway_metagenomics_crew)
pathway_metagenomics_crew.head()

ValueError: The truth value of a DataFrame is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

ValueError: The truth value of a DataFrame is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

### Gene Family Metagenomics

Each column is a microbial swab sample taken from a crew member either during or after flight, from a specific anatomical region.

Each row is a different gene family whose activity was quantified within that sample.

In [13]:
genefam_metagenomics_crew = pd.read_csv(
    'https://osdr.nasa.gov/geode-py/ws/studies/OSD-572/download?source=datamanager'
    '&file=GLDS-564_GMetagenomics_Gene-families-cpm_GLmetagenomics.tsv',
    index_col=0, sep='\t')
genefam_metagenomics_crew = add_col_metadata(genefam_metagenomics_crew)
genefam_metagenomics_crew.head()

ValueError: The truth value of a DataFrame is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

ValueError: The truth value of a DataFrame is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

# 2. Blood Serum Metabolic Panel

Dataset ID: [OSD-575](https://osdr.nasa.gov/bio/repo/data/studies/OSD-575)

The crew collected biospecimen samples before, during, and after flight. One of these biospecimen collections included whole blood collected via venipuncture, with serum extracted from blood using a serum separator tube (SST). Samples were collected pre-flight (L-92, L-44, L-3) and post-flight (R+1, R+45, R+82). Serum samples were submitted to Quest Diagnostics for comprehensive metabolic panel testing.

Each column is a serum sample taken from a crew member.

Each row is a metabolic measurement from Quest.

In [14]:
metabolic_blood = pd.read_csv(
    'https://osdr.nasa.gov/geode-py/ws/studies/OSD-575/download?source=datamanager'
    '&file=LSDS-8_Comprehensive_Metabolic_Panel_CMP_TRANSFORMED.csv',
    index_col=0).transpose()
metabolic_blood = tag_long(metabolic_blood)
metabolic_blood.head()

Sample Name,C001_serum_L-3,C001_serum_L-44,C001_serum_L-92,C001_serum_R+1,C001_serum_R+194,C001_serum_R+45,C001_serum_R+82,C002_serum_L-3,C002_serum_L-44,C002_serum_L-92,...,C004_serum_L-3,C004_serum_L-44,C004_serum_L-92,C004_serum_R+1,C004_serum_R+194,C004_serum_R+45,C004_serum_R+82,crew_id,timepoint,phase
albumin_value_gram_per_deciliter,4.9,5.0,5.0,4.9,4.6,4.9,5.1,4.3,4.6,4.9,...,4.6,4.4,4.6,4.5,4.3,4.5,4.6,UNKNOWN,UNKNOWN,UNKNOWN
albumin_range_min_gram_per_deciliter,3.6,3.6,3.6,3.6,3.6,3.6,3.6,3.6,3.6,3.6,...,3.6,3.6,3.6,3.6,3.6,3.6,3.6,UNKNOWN,UNKNOWN,UNKNOWN
albumin_range_max_gram_per_deciliter,5.1,5.1,5.1,5.1,5.1,5.1,5.1,5.1,5.1,5.1,...,5.1,5.1,5.1,5.1,5.1,5.1,5.1,UNKNOWN,UNKNOWN,UNKNOWN
albumin_to_globulin_ratio_value,2.0,1.9,2.2,1.9,1.9,2.1,1.9,1.6,1.5,1.6,...,1.7,1.5,1.6,1.6,1.5,1.6,1.6,UNKNOWN,UNKNOWN,UNKNOWN
albumin_to_globulin_ratio_range_min,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,UNKNOWN,UNKNOWN,UNKNOWN


# 3. Immune/Cardiac Cytokine Arrays

Dataset ID: [OSD-575](https://osdr.nasa.gov/bio/repo/data/studies/OSD-575)

The crew collected biospecimen samples before, during, and after flight. One of these biospecimen collections included whole blood collected via venipuncture, with serum extracted from blood using a serum separator tube (SST). Samples were collected pre-flight (L-92, L-44, L-3) and post-flight (R+1, R+45, R+82). Serum samples were submitted for immune and cardiovascular cytokine biomarker profiling panels at two different companies (Eve Technologies and Alamar).

Below we read in 3 different files:

* Immune cytokine biomarkers quantified by Eve Technologies
* Immune cytokine biomakers quantified by Alamar
* Cardiac cytokine biomarkers quantified by Eve Technologies


In each file, the columns are serum samples from crew members, and the rows are cytokine measurements from Eve or Alamar.

In [15]:
immune_eve = pd.read_csv(
    'https://osdr.nasa.gov/geode-py/ws/studies/OSD-575/download?source=datamanager'
    '&file=LSDS-8_Multiplex_serum_immune_EvePanel_TRANSFORMED.csv',
    index_col=0).transpose()
immune_eve = tag_long(immune_eve)
immune_eve.head()

Sample Name,C001_serum_L-3,C001_serum_L-44,C001_serum_L-92,C001_serum_R+1,C001_serum_R+194,C001_serum_R+45,C001_serum_R+82,C002_serum_L-3,C002_serum_L-44,C002_serum_L-92,...,C004_serum_L-3,C004_serum_L-44,C004_serum_L-92,C004_serum_R+1,C004_serum_R+194,C004_serum_R+45,C004_serum_R+82,crew_id,timepoint,phase
6ckine_concentration_picogram_per_milliliter,909.520000,844.400000,862.460000,869.650000,598.210000,687.810000,1524.500000,1256.050000,1022.790000,1434.760000,...,601.410000,953.250000,595.000000,601.410000,520.910000,462.100000,1051.880000,UNKNOWN,UNKNOWN,UNKNOWN
6ckine_percent,87.230109,80.984590,82.716686,83.406263,57.373036,65.966380,146.211520,120.465057,98.093592,137.604749,...,57.679941,91.424160,57.065172,57.679941,49.959359,44.319018,100.883551,UNKNOWN,UNKNOWN,UNKNOWN
bca_1_concentration_picogram_per_milliliter,72.490000,49.830000,68.350000,79.860000,41.520000,27.200000,31.410000,31.910000,27.630000,38.010000,...,42.310000,31.260000,38.210000,35.160000,38.840000,28.110000,30.450000,UNKNOWN,UNKNOWN,UNKNOWN
bca_1_percent,144.748403,99.500799,136.481629,159.464856,82.907348,54.313099,62.719649,63.718051,55.171725,75.898562,...,84.484824,62.420128,76.297923,70.207668,77.555911,56.130192,60.802716,UNKNOWN,UNKNOWN,UNKNOWN
ctack_concentration_picogram_per_milliliter,1439.510000,1134.000000,1054.150000,1434.980000,1412.380000,1263.070000,1386.270000,1078.390000,979.560000,1167.010000,...,1015.100000,989.100000,962.490000,941.690000,1419.210000,1027.100000,1217.500000,UNKNOWN,UNKNOWN,UNKNOWN


In [16]:
immune_alamar = pd.read_csv(
    'https://osdr.nasa.gov/geode-py/ws/studies/OSD-575/download?source=datamanager'
    '&file=LSDS-8_Multiplex_serum.immune.AlamarPanel_TRANSFORMED.csv',
    index_col=0).transpose()
immune_alamar = tag_long(immune_alamar)
immune_alamar.head()

Sample ID,C001_serum_L-92,C002_serum_L-92,C003_serum_L-92,C004_serum_L-92,C001_serum_L-44,C002_serum_L-44,C004_serum_L-44,C001_serum_L-3,C002_serum_L-3,C003_serum_L-3,...,C002_serum_R+82,C003_serum_R+82,C004_serum_R+82,C001_serum_R+194,C002_serum_R+194,C003_serum_R+194,C004_serum_R+194,crew_id,timepoint,phase
activin_a_concentration_npq,12.715156,12.771871,12.352381,12.596060,13.002848,12.860655,12.732467,12.760220,12.670691,12.559134,...,12.683116,12.416485,12.592203,12.638560,12.654878,12.350908,12.659547,UNKNOWN,UNKNOWN,UNKNOWN
activin_a_percent_normalized_value,99.135210,100.032362,99.170051,99.504397,101.378233,100.727739,100.581969,99.486557,99.239899,100.829949,...,99.337212,99.684699,99.473931,98.538016,99.116046,99.158219,100.005919,UNKNOWN,UNKNOWN,UNKNOWN
activin_ab_concentration_npq,12.182457,12.263028,11.991739,12.233864,12.458077,12.119235,12.200485,12.181165,12.073917,12.037320,...,12.067680,12.068117,12.012614,12.192767,12.188928,12.305882,12.464335,UNKNOWN,UNKNOWN,UNKNOWN
activin_ab_percent_normalized_value,99.254982,100.913161,99.810309,100.268603,101.500561,99.729882,99.995033,99.244458,99.356957,100.189691,...,99.305634,100.446027,98.455246,99.338980,100.303387,102.425008,102.157541,UNKNOWN,UNKNOWN,UNKNOWN
activin_b_concentration_npq,7.375708,7.867645,8.401332,9.536772,7.612082,8.065223,10.187079,7.830344,7.895528,8.239994,...,7.842926,8.435867,9.166151,7.462066,7.705942,8.051075,8.890991,UNKNOWN,UNKNOWN,UNKNOWN


In [17]:
cardio_eve = pd.read_csv(
    'https://osdr.nasa.gov/geode-py/ws/studies/OSD-575/download?source=datamanager'
    '&file=LSDS-8_Multiplex_serum_cardiovascular_EvePanel_TRANSFORMED.csv',
    index_col=0).transpose()
cardio_eve = tag_long(cardio_eve)
cardio_eve.head()

Sample Name,C001_serum_L-3,C001_serum_L-44,C001_serum_L-92,C001_serum_R+1,C001_serum_R+194,C001_serum_R+45,C001_serum_R+82,C002_serum_L-3,C002_serum_L-44,C002_serum_L-92,...,C004_serum_L-3,C004_serum_L-44,C004_serum_L-92,C004_serum_R+1,C004_serum_R+194,C004_serum_R+45,C004_serum_R+82,crew_id,timepoint,phase
a2_macroglobulin_concentration_nanogram_per_milliliter,1.002300e+06,6.579436e+05,9.037427e+05,1.048700e+06,1.651200e+06,1.130800e+06,2.993600e+06,1.892200e+06,7.342433e+05,2.036900e+06,...,1.780800e+06,6.340179e+05,1.354300e+06,1.651200e+06,1.568600e+06,1.526000e+06,1.785900e+06,UNKNOWN,UNKNOWN,UNKNOWN
a2_macroglobulin_percent,6.870291e+01,4.509891e+01,6.194728e+01,7.188341e+01,1.131819e+02,7.751098e+01,2.051971e+02,1.297013e+02,5.032890e+01,1.396198e+02,...,1.220654e+02,4.345892e+01,9.283084e+01,1.131819e+02,1.075201e+02,1.046001e+02,1.224150e+02,UNKNOWN,UNKNOWN,UNKNOWN
agp_concentration_nanogram_per_milliliter,1.214100e+06,1.745700e+06,1.048700e+06,1.034400e+06,2.595800e+06,1.285200e+06,2.455500e+06,3.653700e+06,1.963300e+06,2.932000e+06,...,3.606700e+06,2.165100e+06,2.918800e+06,3.338700e+06,4.564200e+06,3.484600e+06,2.651400e+06,UNKNOWN,UNKNOWN,UNKNOWN
agp_percent,5.650503e+01,8.124605e+01,4.880720e+01,4.814167e+01,1.208103e+02,5.981407e+01,1.142806e+02,1.700456e+02,9.137330e+01,1.364572e+02,...,1.678582e+02,1.007652e+02,1.358429e+02,1.553853e+02,2.124209e+02,1.621756e+02,1.233979e+02,UNKNOWN,UNKNOWN,UNKNOWN
crp_concentration_picogram_per_milliliter,5.131700e+06,2.191000e+06,1.838500e+07,1.063760e+07,1.541500e+07,6.713900e+06,2.503000e+07,1.196700e+07,1.568500e+06,8.927100e+06,...,1.374200e+07,1.289600e+06,2.137800e+07,1.770100e+07,4.600600e+07,2.240700e+07,7.902300e+06,UNKNOWN,UNKNOWN,UNKNOWN


# 4. Dragon Capsule Metagenomic Microbial Swabs

Dataset ID: [OSD-573](https://osdr.nasa.gov/bio/repo/data/studies/OSD-573)

The crew collected biospecimen samples before, during, and after flight. One of these biospecimen collections included microbial swabs collected from nine surfaces in the Dragon capsule (execute button, G-meter button, control touch screen - left, control touch screen - right, side hatch mobility aid, lid of waste locker, seat 2, commode panel, viewing dome) and one open air control. Swabs were collected twice during flight (flight day 2 (FD2), and flight day 3 (FD3)) and twice pre-flight in the crew training capsule in Hawthorne, CA (L-92, L-44).

There are several different ways to quantify how the microbial populations are changing in each swab at each timepoint. Below we read in 4 different files which quantify 4 different aspects of the microbial population in each swab:

* microbial function using [KEGG orthology](https://www.genome.jp/kegg/ko.html)
* microbial taxon abundance
* molecular pathway activity
* gene family abundance


More information on the metagenomics bioinformatics pipeline that generated these files is available [here](https://github.com/nasa/GeneLab_Data_Processing/tree/master/Metagenomics/Illumina).

### KEGG Orthology (KO) Metagenomics

Each column is a microbial swab sample taken from a region of the Dragon Capsule pre- or during flight.

Each row is a [KEGG orthology](https://www.genome.jp/kegg/ko.html) molecular function that has been quantified within that sample.

In [43]:
kegg_metagenomics_dragon = pd.read_csv(
    'https://osdr.nasa.gov/geode-py/ws/studies/OSD-573/download?source=datamanager'
    '&file=GLDS-565_GMetagenomics_Combined-gene-level-KO-function-coverages-CPM_GLmetagenomics.tsv',
    index_col=0, sep='\t')
kegg_metagenomics_dragon = add_col_metadata(kegg_metagenomics_dragon)
kegg_metagenomics_dragon.head()

ValueError: The truth value of a DataFrame is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

ValueError: The truth value of a DataFrame is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

### Taxonomy Metagenomics

Each column is a microbial swab sample taken from a region of the Dragon Capsule pre- or during flight.

Each row is a different microbial taxon that has been quantified within that sample.

In [44]:
tax_metagenomics_dragon = pd.read_csv(
    'https://osdr.nasa.gov/geode-py/ws/studies/OSD-573/download?source=datamanager'
    '&file=GLDS-565_GMetagenomics_Combined-gene-level-taxonomy-coverages-CPM_GLmetagenomics.tsv',
    index_col=0, sep='\t')
tax_metagenomics_dragon = add_col_metadata(tax_metagenomics_dragon)
tax_metagenomics_dragon.head()

ValueError: The truth value of a DataFrame is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

ValueError: The truth value of a DataFrame is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

### Pathway Metagenomics

Each column is a microbial swab sample taken from a region of the Dragon Capsule pre- or during flight.

Each row is a different molecular pathway whose activity was quantified within that sample.

In [20]:
pathway_metagenomics_dragon = pd.read_csv(
    'https://osdr.nasa.gov/geode-py/ws/studies/OSD-573/download?source=datamanager'
    '&file=GLDS-565_GMetagenomics_Pathway-abundances-cpm_GLmetagenomics.tsv',
    index_col=0, sep='\t')
pathway_metagenomics_dragon = add_col_metadata(pathway_metagenomics_dragon)
pathway_metagenomics_dragon.head()

ValueError: The truth value of a DataFrame is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

ValueError: The truth value of a DataFrame is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

### Gene Family Metagenomics

Each column is a microbial swab sample taken from a region of the Dragon Capsule pre- or during flight.

Each row is a different gene family whose activity was quantified within that sample.

In [21]:
genefam_metagenomics_dragon = pd.read_csv(
    'https://osdr.nasa.gov/geode-py/ws/studies/OSD-573/download?source=datamanager'
    '&file=GLDS-565_GMetagenomics_Gene-families-cpm_GLmetagenomics.tsv',
    index_col=0, sep='\t')
genefam_metagenomics_dragon = add_col_metadata(genefam_metagenomics_dragon)
genefam_metagenomics_dragon.head()

ValueError: The truth value of a DataFrame is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

ValueError: The truth value of a DataFrame is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

# 5. Metabolomics

Dataset ID: [OSD-571](https://osdr.nasa.gov/bio/repo/data/studies/OSD-571)

The crew collected biospecimen samples before, during, and after flight. One of these biospecimens was plasma from venous blood, which was collected pre-flight (L-92, L-44, L-3) and post-flight (R+1, R+45, R+82). Plasma was used for metabolomics; metabolite abundances were calculated pre vs post-flight to generate processed data files.

Notes on processed data:

* Mission	Inspiration4
* Data	Plasma Metabolomics
* Database Identifier	I4-FP1
* Comparison	(R+1) vs (L-92, L-44, L-3)
* Data Note 1:	Positive logFC = higher abundance post-flight; Negative logFC = lower abundance post-flight.
* Data Note 2:	Calculated with limma (version 3.52)


The columns are statistics from limma differential expression; the rows are proteins.

In [22]:
metabolomics = pd.read_excel(
    'https://osdr.nasa.gov/geode-py/ws/studies/OSD-571/download?source=datamanager'
    '&file=GLDS-563_metabolomics_Plasma_Metabolomics_Processed_Data.xlsx',
    skiprows=[0,1,2,3,4,5], index_col=0)
metabolomics.head()

,logFC,AveExpr,t,P.Value,adj.P.Val,B
ID,,,,,,
LysoPC(15:0),-2.341561,2.711373e-15,-6.721362,5.614970e-08,0.000037,8.306020
LysoPC(14:0),-2.108057,2.225650e-15,-5.938794,6.681835e-07,0.000217,5.943995
LysoPC(17:0),-2.046564,1.492151e-15,-5.813748,9.936616e-07,0.000217,5.565445
Inosine,2.160162,-1.476151e-01,5.685773,1.491301e-06,0.000245,5.178206
Sphingosine 1-phosphate,-1.907023,1.637868e-15,-5.339793,4.460217e-06,0.000585,4.133896


# 6. EVPs (Extracellular Vesicles and Particles) for Mass Spectrometry

Dataset ID: [OSD-571](https://osdr.nasa.gov/bio/repo/data/studies/OSD-571)

The crew collected biospecimen samples before, during, and after flight. One of these biospecimens was plasma from venous blood, which was collected pre-flight (L-92, L-44, L-3) and post-flight (R+1, R+45, R+82). Plasma was used to isolate extracellular vesicles and particles (EVPs) for mass spectrometry.

Notes on processed data:

* Comparison	(R+1) vs (L-92, L-44, L-3)
* Data Note 1:	Positive logFC = higher abundance post-flight; Negative logFC = lower abundance post-flight.
* Data Note 2:	Calculated with limma (version 3.52)


The columns are statistics from limma differential expression; the rows are proteins quantified from the EVPs through mass spectrometry.



In [23]:
evp = pd.read_excel(
    'https://osdr.nasa.gov/geode-py/ws/studies/OSD-571/download?source=datamanager'
    '&file=GLDS-563_proteomics_EVP_Proteomics_Processed_Data.xlsx',
    skiprows=[0,1,2,3,4,5], index_col=0)
evp.head()

,logFC,AveExpr,t,P.Value,adj.P.Val,B
Gene,,,,,,
CD58,3.344275,24.769393,8.261797,4.149066e-08,0.000022,8.830804
TFRC,-1.953815,29.189652,-7.196199,3.780941e-07,0.000095,6.707331
ART4,2.087256,24.074245,7.029155,5.420177e-07,0.000095,6.359426
AHCY,1.984654,23.972799,6.856379,7.897293e-07,0.000098,5.995405
FN1,-2.384021,31.939915,-6.780853,9.321097e-07,0.000098,5.834963


# 7. Proteomics

Dataset ID: [OSD-571](https://osdr.nasa.gov/bio/repo/data/studies/OSD-571)

The crew collected biospecimen samples before, during, and after flight. One of these biospecimens was plasma from venous blood, which was collected pre-flight (L-92, L-44, L-3) and post-flight (R+1, R+45, R+82). Plasma was used for proteomics; differential protein abundances were calculated pre vs post-flight to generate processed data files.

Notes on processed data:

* Comparison	(R+1) vs (L-92, L-44, L-3)
* Data Note 1:	Positive logFC = higher abundance post-flight; Negative logFC = lower abundance post-flight.
* Data Note 2:	Calculated with limma (version 3.52)


The columns are statistics from limma differential expression; the rows are protein abundances.


In [24]:
protein = pd.read_excel(
    'https://osdr.nasa.gov/geode-py/ws/studies/OSD-571/download?source=datamanager'
    '&file=GLDS-563_proteomics_Plasma_Proteomics_Processed_Data.xlsx',
    skiprows=[0,1,2,3,4,5], index_col=0)
protein.head()

,logFC,AveExpr,t,P.Value,adj.P.Val,B
Gene,,,,,,
PF4,1.445671,18.630286,7.957981,2.589947e-07,0.000457,7.043966
QPCT,-1.604759,10.699338,-6.981444,1.584531e-06,0.001045,5.373022
TGFB1,1.483192,16.085805,6.922152,1.776072e-06,0.001045,5.266779
APP,1.320346,12.910695,6.725544,2.601841e-06,0.001148,4.910578
CCN2,1.277208,13.223742,6.366990,5.290941e-06,0.001868,4.245643


# 8. Urine Inflammation Panel (Multiplex, NULISAseq)

Dataset ID: [OSD-656](https://osdr.nasa.gov/bio/repo/data/studies/OSD-656)

The crew collected biospecimen samples before, during, and after flight. One of these biospecimen collections included urine from pre-flight and post-flight/recovery timepoints (L-92, L-44, L-3, R+1, R+45, R+82). 203 inflammatory, cytokine, and chemokine proteins were quantified using NULISAseq. This study derives results from the Multiplex assay.

The columns are quantifications of inflammatory proteins; the rows are samples from crew members at specific time points.

In [25]:
urine = pd.read_csv(
    'https://osdr.nasa.gov/geode-py/ws/studies/OSD-656/download?source=datamanager'
    '&file=LSDS-64_Multiplex_urine.immune.AlamarPanel_TRANSFORMED.csv',
    index_col=0)
urine = tag_long(urine)
urine.head()

,activin_a_concentration_npq,Unnamed: 2,activin_ab_concentration_npq,activin_ab_percent_normalized_value,activin_b_concentration_npq,activin_b_percent_normalized_value,ager_concentration_npq,ager_percent_normalized_value,agrp_concentration_npq,agrp_percent_normalized_value,...,vsnl1_percent_normalized_value,vstm1_concentration_npq,vstm1_percent_normalized_value,wnt16_concentration_npq,wnt16_percent_normalized_value,wnt7a_concentration_npq,wnt7a_percent_normalized_value,crew_id,timepoint,phase
Sample Name,,,,,,,,,,,,,,,,,,,,,
C001_urine_L-92,12.746733,99.821962,11.815756,101.348395,7.224334,98.366855,3.141673,101.327660,18.790892,103.913740,...,100.575818,14.791355,100.499494,3.475709,64.577199,10.697436,98.029902,C001,L-92,Preflight
C002_urine_L-92,12.827067,100.443652,11.192127,95.527208,7.703489,103.216448,6.322451,147.306775,21.143020,107.966275,...,109.291871,10.661033,108.880456,2.828991,69.638032,10.973062,99.829057,C002,L-92,Preflight
C003_urine_L-92,12.572543,99.192139,12.312586,101.420165,7.419941,100.225422,11.199742,99.619818,20.292069,110.736312,...,97.956147,14.833864,98.527904,3.717082,107.182287,11.042905,102.255206,C003,L-92,Preflight
C004_urine_L-92,12.720792,99.896522,11.070407,95.493127,7.051414,96.817483,2.977937,205.951726,22.379647,116.814931,...,109.567230,16.972360,110.479293,6.055749,147.263903,10.721241,99.819408,C004,L-92,Preflight
C001_urine_L-44,12.792202,100.178038,11.501349,98.651605,7.464220,101.633145,3.059344,98.672340,17.375436,96.086260,...,99.424182,14.644325,99.500506,7.288800,135.422801,11.127407,101.970098,C001,L-44,Preflight


# 9. Stool Metagenome Profiling

Dataset ID: [OSD-630](https://osdr.nasa.gov/bio/repo/data/studies/OSD-630)

The crew collected biospecimen samples before, during, and after flight. One of these biospecimen collections included stool collected in OMNIgene•GUT tubes (DNA Genotek, OMR-200). Samples were collected pre-flight (L-92, L-44) and post-flight (R+45, R+82).

There are several different ways to quantify how the microbial populations are changing in each sample at each timepoint. Below we read in 4 different files which quantify 4 different aspects of the microbial population in each sample:

* microbial function using [KEGG orthology](https://www.genome.jp/kegg/ko.html)
* microbial taxon abundance
* molecular pathway activity
* gene family abundance


More information on the metagenomics bioinformatics pipeline that generated these files is available [here](https://github.com/nasa/GeneLab_Data_Processing/tree/master/Metagenomics/Illumina).

### KEGG Orthology (KO) Metagenomics

Each column is a stool sample from a crew member pre- or during flight.

Each row is a [KEGG orthology](https://www.genome.jp/kegg/ko.html) molecular function that has been quantified within that sample.

In [26]:
kegg_metagenomics_stool = pd.read_csv(
    'https://osdr.nasa.gov/geode-py/ws/studies/OSD-630/download?source=datamanager'
    '&file=GLDS-599_GMetagenomics_Combined-gene-level-KO-function-coverages-CPM_GLmetagenomics.tsv',
    index_col=0, sep='\t')
kegg_metagenomics_stool = add_col_metadata(kegg_metagenomics_stool)
kegg_metagenomics_stool.head()

,KO_function,C002_L-44,C002_L-92,C002_R+45,C002_R+82,C004_L-44,C004_L-92,C004_R+45,C004_R+82
KO_ID,,,,,,,,,
K00003,homoserine dehydrogenase [EC:1.1.1.3],230.467819,289.371067,291.177791,208.775172,205.655940,246.399989,210.994407,166.923566
K00004,"(R,R)-butanediol dehydrogenase / meso-butanedi...",0.000000,0.000000,0.000000,0.000000,0.000000,6.416817,0.000000,0.000000
K00005,glycerol dehydrogenase [EC:1.1.1.6],65.841934,70.643844,107.220348,57.103263,37.329941,41.210053,41.186732,22.492698
K00006,glycerol-3-phosphate dehydrogenase (NAD+) [EC:...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.860091
K00008,L-iditol 2-dehydrogenase [EC:1.1.1.14],12.996759,26.749632,16.034788,14.743143,5.142672,6.689531,4.103665,4.235197


### Taxonomy Metagenomics

Each column is a stool sample from a crew member pre- or during flight.

Each row is a different microbial taxon that has been quantified within that sample.

In [27]:
tax_metagenomics_stool = pd.read_csv(
    'https://osdr.nasa.gov/geode-py/ws/studies/OSD-630/download?source=datamanager'
    '&file=GLDS-599_GMetagenomics_Combined-gene-level-taxonomy-coverages-CPM_GLmetagenomics.tsv',
    index_col=0, sep='\t')
tax_metagenomics_stool = add_col_metadata(tax_metagenomics_stool)
tax_metagenomics_stool.head()

,domain,phylum,class,order,family,genus,species,C002_L-44,C002_L-92,C002_R+45,C002_R+82,C004_L-44,C004_L-92,C004_R+45,C004_R+82
taxid,,,,,,,,,,,,,,,
100134,Bacteria,Firmicutes,Clostridia,Clostridiales,Lachnospiraceae,Anaerocolumna,Anaerocolumna xylanovorans,0.000000,13.396411,8.744940,0.000000,0.000000,0.000000,0.000000,0.000000
100176,Bacteria,Firmicutes,Clostridia,Clostridiales,Ruminococcaceae,Papillibacter,Papillibacter cinnamivorans,7.208219,1.795719,6.337918,3.642914,0.000000,2.955006,5.537474,3.652669
1002367,Bacteria,Bacteroidetes,Bacteroidia,Bacteroidales,Prevotellaceae,Prevotella,Prevotella stercorea,2.378330,0.000000,4.675497,4.267181,0.000000,3.230138,0.000000,0.000000
1002795,Bacteria,Firmicutes,Negativicutes,Selenomonadales,Selenomonadaceae,Pectinatus,Pectinatus sottacetonis,0.000000,0.000000,0.000000,0.000000,4.739918,3.817086,6.238668,5.061456
1003195,Bacteria,Actinobacteria,Actinobacteria,Streptomycetales,Streptomycetaceae,Streptomyces,Streptomyces cattleya,1.819380,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


### Pathway Metagenomics

Each column is a stool sample from a crew member pre- or during flight.

Each row is a different molecular pathway whose activity was quantified within that sample.

In [28]:
pathway_metagenomics_stool = pd.read_csv(
    'https://osdr.nasa.gov/geode-py/ws/studies/OSD-630/download?source=datamanager'
    '&file=GLDS-599_GMetagenomics_Pathway-abundances-cpm_GLmetagenomics.tsv',
    index_col=0, sep='\t')
pathway_metagenomics_stool = add_col_metadata(pathway_metagenomics_stool)
pathway_metagenomics_stool.head()

,C002_L-44_Abundance-CPM,C002_L-92_Abundance-CPM,C002_R+45_Abundance-CPM,C002_R+82_Abundance-CPM,C004_L-44_Abundance-CPM,C004_L-92_Abundance-CPM,C004_R+45_Abundance-CPM,C004_R+82_Abundance-CPM
# Pathway,,,,,,,,
UNMAPPED,209476.00000,286500.00000,313172.00000,277538.0000,321989.00000,353632.000,405313.000,386243.000
UNINTEGRATED,739727.00000,659969.00000,631741.00000,671919.0000,626978.00000,597464.000,545843.000,566050.000
1CMET2-PWY: folate transformations III (E. coli),420.29200,446.90500,452.50400,425.9220,394.24700,380.864,368.998,346.280
ALLANTOINDEG-PWY: superpathway of allantoin degradation in yeast,3.55071,4.88755,2.03556,2.1202,4.33417,0.000,0.000,0.000
ANAEROFRUCAT-PWY: homolactic fermentation,230.82700,318.79300,338.62200,335.4170,371.03400,343.525,366.040,360.538


### Gene Family Metagenomics

Each column is a stool sample from a crew member pre- or during flight.

Each row is a different gene family whose activity was quantified within that sample.

In [29]:
genefam_metagenomics_stool = pd.read_csv(
    'https://osdr.nasa.gov/geode-py/ws/studies/OSD-630/download?source=datamanager'
    '&file=GLDS-599_GMetagenomics_Gene-families-cpm_GLmetagenomics.tsv',
    index_col=0, sep='\t')
genefam_metagenomics_stool = add_col_metadata(genefam_metagenomics_stool)
genefam_metagenomics_stool.head()

,C002_L-44_Abundance-CPM,C002_L-92_Abundance-CPM,C002_R+45_Abundance-CPM,C002_R+82_Abundance-CPM,C004_L-44_Abundance-CPM,C004_L-92_Abundance-CPM,C004_R+45_Abundance-CPM,C004_R+82_Abundance-CPM
# Gene Family,,,,,,,,
UNMAPPED,209476.0,286500.000000,313172.000000,277538.00000,321989.000000,353632.000000,405313.000000,386243.000000
UniRef90_A0A010YTZ2,0.0,0.298162,0.000000,0.00000,0.740116,0.288805,0.346212,0.674229
UniRef90_A0A010Z266,0.0,0.000000,0.000000,0.00000,0.346077,0.000000,0.000000,0.000000
UniRef90_A0A011A4H8,0.0,0.000000,0.369799,0.00000,0.000000,0.000000,0.000000,0.000000
UniRef90_A0A011A697,0.0,0.000000,0.000000,0.55435,0.000000,0.000000,0.000000,0.000000


# 10. Whole Blood Profiling

Dataset ID: [OSD-569](https://osdr.nasa.gov/bio/repo/data/studies/OSD-569)

The crew collected biospecimen samples before, during, and after flight. One of these biospecimen collections included whole blood collected via venipuncture, collected pre-flight (L-92, L-44, L-3) and post-flight (R+1, R+45, R+82, R+194). Total RNA was purified from each tube and mRNA was sequenced. Data was used to generate differential gene expression profiles and annotate sites of m6A modification. Also, whole blood was submitted to Quest Diagnostics for a complete blood count at all ground timepoints (L-92, L-44, L-3, R+1, R+45, R+82, R+194).

### Total RNA Sequencing

Notes on processed data:
*   Comparison: (R+82) vs (L-92, L-44, L-3)
*   Data Note 1: FeatureCounts was used to calculate DEGs with DESeq2. Salmon was used with pipeline-transcriptome-de.
*   Data Note 2: Positive logFC = higher abundance at (R+82); Negative logFC = lower abundance at (R+82).


The rows are genes; the columns are samples from crew members at specific time poionts.




In [42]:
rna_blood = pd.read_excel(
    "https://osdr.nasa.gov/geode-py/ws/studies/OSD-569/download?source=datamanager"
    "&file=GLDS-561_long-readRNAseq_Direct_RNA_seq_Gene_Expression_Processed.xlsx",
    skiprows=[0,1,2,3,4,5,6,9], header=[0,1], index_col=0)
rna_blood.columns = [
    f"{str(a)}_{str(b)}".strip("_") for a, b in rna_blood.columns
]
rna_blood = add_col_metadata(rna_blood)
rna_blood.head()

ValueError: The truth value of a DataFrame is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

ValueError: The truth value of a DataFrame is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

### m6A Modification

m6A is a chemical modification on mRNA that affects how transcripts are regulated, degraded, and translated. Measuring it in whole blood across timepoints reveals how spaceflight stress alters post-transcriptional gene regulation.

Notes on processed data:
* Comparison: (R+1) vs (L-92, L-44, L-3)
* Data Note 1: Calculated with m6Anet (v1.1.1) and methyl-kit (v1.22.0)
* Data Note 2: Position refers to the location on the transcript, relative from the 5' transcriptional start site.
* Data Note 3: Crew member/timepoint columns refers to the probability that within the indicated sample the position within that particular transcript contains an m6a modification. NA denotes positions within samples that lacked coverage to make a prediction (i.e. less than twenty reads per the definition of m6Anet).
* Data Note 4: Higher or lower abundance postflight is indicated by positive or negative methylKit methylation differences, respectively, for the given comparison (indicated within column W).


The rows are gene and transcript names; the columns are samples from crew members at specific time points.


In [31]:
m6A = pd.read_excel(
    "https://osdr.nasa.gov/geode-py/ws/studies/OSD-569/download?source=datamanager"
    "&file=GLDS-561_directm6Aseq_Direct_RNA_seq_m6A_Processed_Data.xlsx",
    skiprows=[0,1,2,3,4,5,6,7], header=[0,1], index_col=0)
m6A.columns = [
    f"{str(a)}_{str(b)}".strip("_") for a, b in m6A.columns
]
m6A.index.name = "transcript_ENSEMBL"
m6A = add_col_metadata(m6A)
m6A.head()

ValueError: The truth value of a DataFrame is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

ValueError: The truth value of a DataFrame is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

### Complete Blood Count (CBC)

Each row is a measurement from the CBC assay. Each column are the values and units, as well as the crew member (SUBJECT_ID) and time point (TEST_DATE).


In [32]:
cbc = pd.read_csv(
    'https://osdr.nasa.gov/geode-py/ws/studies/OSD-569/download?source=datamanager'
    '&file=LSDS-7_Complete_Blood_Count_CBC.upload_SUBMITTED.csv',
    index_col=0)
cbc = tag_long(cbc, crew_col="SUBJECT_ID", timepoint_col="TEST_DATE")
cbc.head()

,VALUE,RANGE_MIN,RANGE_MAX,UNITS,TEST_TYPE,SUBJECT_ID,SEX,TEST_DATE,crew_id,timepoint,phase
ANALYTE,,,,,,,,,,,
WHITE BLOOD CELL COUNT,5.40,3.8,10.8,Thousand/uL,CBC,C001,M,L-92,C001,L-92,Preflight
RED BLOOD CELL COUNT,4.58,4.2,5.8,Million/uL,CBC,C001,M,L-92,C001,L-92,Preflight
HEMOGLOBIN,14.10,13.2,17.1,g/dL,CBC,C001,M,L-92,C001,L-92,Preflight
HEMATOCRIT,42.50,38.5,50.0,%,CBC,C001,M,L-92,C001,L-92,Preflight
MCV,92.80,80.0,100.0,fL,CBC,C001,M,L-92,C001,L-92,Preflight


# 11. PBMC (Peripheral Blood Mononuclear Cells) Profiling

Dataset ID: [OSD-570](https://osdr.nasa.gov/bio/repo/data/studies/OSD-570)

The crew collected biospecimen samples before, during, and after flight. One of these biospecimen collections included PBMCs collected via venipuncture, collected pre-flight (L-92, L-44, L-3) and post-flight (R+1, R+45, R+82). Single-nuclei RNA-seq and single-nuclei ATAC-seq were co-assayed (L-92, L-44, L-3, R+1, R+45, R+82 timepoints). T-cell and B-cell V(D)J repertoires were profiled (L-3, R+1, R+45, R+82 timepoints). Differential gene expression profiles and clonotypes detected per timepoint were calculated from sequencing results.


PBMCs are  the immune cells circulating in blood, and profiling them reveals how the immune system is adapting or dysregulating in real time.

### Single-nuclei RNA-seq

Notes on processed data:

* Comparison:	(R+45) vs (R+1)
* Data Note 1:	Output of Seurat (v4.2.0) FindMarkers with a log fold change (avg_log2FC) cutoff point of 0.25.
* Data Note 2:	FindMarkers was calculated separately for each cell type. All cell types have been appended to this sheet.
* Data Note 3:	Positive avg_log2FC = higher gene expression at (R+45); Negative avg_log2FC = lower gene expression at (R+45).

The rows are genes; the columns are statistics from the FindMarkers analysis.

In [33]:
snrnaseq = pd.read_excel(
    'https://osdr.nasa.gov/geode-py/ws/studies/OSD-570/download?source=datamanager'
    '&file=GLDS-562_snRNA-Seq_PBMC_Gene_Expression_snRNA-seq_Processed_Data.xlsx',
    skiprows=[0,1,2,3,4,5,6], index_col=0)
snrnaseq.head()

,p_val,avg_log2FC,pct.1,pct.2,p_val_adj,Cell Type
Gene,,,,,,
PF4,0.0,0.269197,0.183,0.002,0.0,B Cell
PPBP,0.0,0.417471,0.263,0.005,0.0,B Cell
RPL41,0.0,-1.967126,0.394,0.976,0.0,B Cell
LYZ,0.0,0.603567,0.457,0.035,0.0,B Cell
PLCG2,0.0,2.182151,0.998,0.892,0.0,B Cell


### Single-nuclei ATAC-seq

Notes on processed data:
* Comparison:	(R+1) vs (L-92, L-44, L-3)
* Data Note 1:	Output of Seurat (v4.2.0) FindMarkers with a log fold change (avg_log2FC) cutoff point of 0.25.
* Data Note 2:	FindMarkers was calculated separately for each cell type. All cell types have been appended to this sheet.
* Data Note 3:	Positive avg_log2FC = higher peak at (R+1); Negative avg_log2FC = lower peak at (R+1).

Each row is a chromatin accessibility peak defined by its chromosome and start/end coordinates; the columns are statistics from the FindMarkers analysis.

In [34]:
snatacseq = pd.read_excel(
    'https://osdr.nasa.gov/geode-py/ws/studies/OSD-570/download?source=datamanager'
    '&file=GLDS-562_snATAC-Seq_PBMC_Chromatin_Accessibility_snATAC-seq_Processed_Data.xlsx',
    skiprows=[0,1,2,3,4,5,6], index_col=0)
snatacseq.head()

,p_val,avg_log2FC,pct.1,pct.2,p_val_adj,Cell Type
Region,,,,,,
chr7-70455925-70457199,1.054123e-32,0.446002,0.157,0.047,2.521399e-27,B Cell
chr10-30284395-30285967,2.893863e-32,0.379825,0.159,0.048,6.921947e-27,B Cell
chr7-95595753-95597173,2.036902e-28,0.347283,0.214,0.083,4.872147e-23,B Cell
chr9-107867976-107868591,2.146729e-28,0.402270,0.129,0.037,5.134848e-23,B Cell
chr14-105836883-105837843,6.708160e-26,0.412864,0.106,0.028,1.604552e-20,B Cell


### T-cell and B-cell VDJ Profiles

V(D)J profiles capture the unique DNA rearrangements that give each T and B cell its specific receptor. Sequencing these from PBMCs tells you which immune cell clones are expanding or contracting over the course of spaceflight, revealing whether the immune system is mounting specific responses, losing diversity, or shifting in ways that could indicate immune dysfunction.


Notes from processed data:
* Data note: Calculated with VGenes package (https://github.com/WilsonImmunologyLab/Vgenes)

Each row is a single TCR clone identified from a crew member at a specific timepoint, with its alpha or beta chain sequence and any somatic mutations relative to the germline; the columns contain additional information and quantification as well as the crew member (crewID) and timepoint.


In [35]:
vdj = pd.read_excel(
    'https://osdr.nasa.gov/geode-py/ws/studies/OSD-570/download?source=datamanager'
    '&file=GLDS-562_scRNA-Seq_VDJ_Results.xlsx',
    skiprows=[0,1,2], index_col=0)
vdj = tag_long(vdj, crew_col="crewID", timepoint_col="timepoint")
vdj.head()

,SeqName,SeqLen,GeneType,Grouping,Species,Sequence,GermlineSequence,ClonalPool,TotalMuts,Mutations,Isotype,crewID,timepoint,crew_id,phase
1,TCR_C001_3_clonetype1_B1,499,Beta,TCR_C1_3,Human,GACACAGCTGTTTCCCAGACTCCAAAATACCTGGTCACACAGATGG...,GACACAGCTGTTTCCCAGACTCCAAAATACCTGGTCACACAGATGG...,14526,0,NaN,Unknown,C001,L-3,C001,Preflight
2,TCR_C001_3_clonetype1_A2,567,Alpha,TCR_C1_3,Human,GCCCAGTCGGTGACCCAGCTTGGCAGCCACGTCTCTGTCTCTGAGG...,GCCCAGTCGGTGACCCAGCTTGGCAGCCACGTCTCTGTCTCTGAAG...,10592,3,A-45-G|T-154-A|C-158-G,Unknown,C001,L-3,C001,Preflight
3,TCR_C001_3_clonetype2_B1,500,Beta,TCR_C1_3,Human,GAACCTGAAGTCACCCAGACTCCCAGCCATCAGGTCACACAGATGG...,GAACCTGAAGTCACCCAGACTCCCAGCCATCAGGTCACACAGATGG...,0,0,NaN,Unknown,C001,L-3,C001,Preflight
4,TCR_C001_3_clonetype2_B2,529,Beta,TCR_C1_3,Human,GAACCTGAAGTCACCCAGACTCCCAGCCATCAGGTCACACAGATGG...,GAACCTGAAGTCACCCAGACTCCCAGCCATCAGGTCACACAGATGG...,12885,0,NaN,Unknown,C001,L-3,C001,Preflight
5,TCR_C001_3_clonetype2_B3,493,Beta,TCR_C1_3,Human,GACACAGCTGTTTCCCAGACTCCAAAATACCTGGTCACACAGATGG...,GACACAGCTGTTTCCCAGACTCCAAAATACCTGGTCACACAGATGG...,0,0,NaN,Unknown,C001,L-3,C001,Preflight


# 12. Deltoid Skin and Microbiome Data

Dataset ID: [OSD-574](https://osdr.nasa.gov/bio/repo/data/studies/OSD-574)

The crew collected biospecimen samples before, during, and after flight. One of these biospecimen collections included deltoid skin biopsies, collected once pre-flight (L-44) and within 1 day postflight (R+1). Samples were used for **spatially resolved transcriptomics** using the NanoString GeoMx Whole Transcriptome Atlas Panel. Skin was profiled in four regions: outer epidermis (OE), inner epidermis (IE), outer dermis (OD), and vasculature (VA) regions. Four crew members were profiled pre-fight (C001, C002, C003, C004) and three crew members were profiled post-flight (C002, C003, C004).

Additionally, skin swabs of the deltoid region were collected before each biopsy. DNA was extracted from the swab to generate **metagenomic profiles**.
There are several different ways to quantify how the microbial populations are changing in each swab at each timepoint. Below we read in 4 different files which quantify 4 different aspects of the microbial population in each swab:

* microbial function using [KEGG orthology](https://www.genome.jp/kegg/ko.html)
* microbial taxon abundance
* molecular pathway activity
* gene family abundance


More information on the metagenomics bioinformatics pipeline that generated these files is available [here](https://github.com/nasa/GeneLab_Data_Processing/tree/master/Metagenomics/Illumina).


### Spatial transcriptomics differential gene expression analysis

Notes on processed data:
* Comparison:	(R+1) vs (L-44)
* Data Note 1:	Positive logFC = higher expression post-flight; Negative logFC = lower expression post-flight.
* Data Note 2:	GeoMx Whole Transcriptome Atlas Panel (WTA, 18,677 genes)
* Data Note 3:	OE = Outer Epidermis, IE = Inner Epidermis, OD = Outer Dermis, VA = Vasculature



In [36]:
spatial = pd.read_excel(
    'https://osdr.nasa.gov/geode-py/ws/studies/OSD-570/download?source=datamanager'
    '&file=GLDS-566_SpatialTranscriptomics_Skin_Biopsy_Spatially_Resolved_Transcriptomics_Processed_Data.xlsx',
    skiprows=[0,1,2,3,4,5,6], index_col=0)
spatial.head()

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
Gene,,,,,,
DES,38.018277,-1.692186,0.355865,-4.755132,1.980000e-06,0.003448
TAGLN,44.887807,-1.548786,0.288775,-5.363304,8.170000e-08,0.000315
GSN,26.085019,-1.421852,0.332504,-4.276196,1.900000e-05,0.008621
ACTG2,41.476678,-1.342798,0.319154,-4.207367,2.580000e-05,0.009931
MYLK,58.374412,-1.298184,0.341808,-3.797994,1.458720e-04,0.023069


### KEGG Orthology (KO) Metagenomics

Each column is a skin biopsy sample from a crew member pre- or post- flight.

Each row is a [KEGG orthology](https://www.genome.jp/kegg/ko.html) molecular function that has been quantified within that sample.

In [37]:
kegg_metagenomics_skin = pd.read_csv(
    'https://osdr.nasa.gov/geode-py/ws/studies/OSD-574/download?source=datamanager'
    '&file=GLDS-566_GMetagenomics_Combined-gene-level-KO-function-coverages-CPM_GLmetagenomics.tsv',
    index_col=0, sep='\t')
kegg_metagenomics_skin = add_col_metadata(kegg_metagenomics_skin)
kegg_metagenomics_skin.head()

,KO_function,C001_L-44_DEL,C001_R+1_DEL,C002_L-44_DEL,C002_R+1_DEL,C003_L-44_DEL,C003_R+1_DEL,C004_L-44_DEL,C004_R+1_DEL
KO_ID,,,,,,,,,
K00003,homoserine dehydrogenase [EC:1.1.1.3],383.429945,331.476280,453.656148,0.0,361.865328,275.137771,324.198740,218.153614
K00004,"(R,R)-butanediol dehydrogenase / meso-butanedi...",21.951350,0.000000,0.000000,0.0,0.000000,0.000000,18.163875,0.000000
K00005,glycerol dehydrogenase [EC:1.1.1.6],25.871233,0.000000,0.000000,0.0,0.000000,0.000000,17.760198,0.000000
K00006,glycerol-3-phosphate dehydrogenase (NAD+) [EC:...,0.000000,31.172965,0.000000,0.0,0.000000,0.000000,41.450057,46.575322
K00010,myo-inositol 2-dehydrogenase / D-chiro-inosito...,422.268076,235.495649,0.000000,0.0,377.943511,266.501749,211.412791,250.311827


### Taxonomy Metagenomics

Each column is a skin biopsy sample from a crew member pre- or post- flight.

Each row is a different microbial taxon that has been quantified within that sample.

In [38]:
tax_metagenomics_skin = pd.read_csv(
    'https://osdr.nasa.gov/geode-py/ws/studies/OSD-574/download?source=datamanager'
    '&file=GLDS-566_GMetagenomics_Combined-gene-level-taxonomy-coverages-CPM_GLmetagenomics.tsv',
    index_col=0, sep='\t')
tax_metagenomics_skin = add_col_metadata(tax_metagenomics_skin)
tax_metagenomics_skin.head()

,domain,phylum,class,order,family,genus,species,C001_L-44_DEL,C001_R+1_DEL,C002_L-44_DEL,C002_R+1_DEL,C003_L-44_DEL,C003_R+1_DEL,C004_L-44_DEL,C004_R+1_DEL
taxid,,,,,,,,,,,,,,,
1003195,Bacteria,Actinobacteria,Actinobacteria,Streptomycetales,Streptomycetaceae,Streptomyces,Streptomyces cattleya,2.299942,0.000000,0.000000,0.0,19.438552,9.683834,19.692014,16.742341
1003237,Bacteria,Proteobacteria,Alphaproteobacteria,Rhodospirillales,Rhodospirillaceae,Nitrospirillum,Nitrospirillum amazonense,0.000000,0.000000,0.000000,0.0,11.139550,0.000000,0.000000,0.000000
100787,Eukaryota,Ascomycota,Sordariomycetes,Glomerellales,Plectosphaerellaceae,Verticillium,Verticillium longisporum,0.000000,54.327434,126.747567,0.0,0.000000,0.000000,47.539395,33.346203
1008454,Bacteria,Firmicutes,Bacilli,Bacillales,Staphylococcaceae,Staphylococcus,Staphylococcus epidermidis,34.260625,0.000000,0.000000,0.0,0.000000,0.000000,11.474824,0.000000
100861,Eukaryota,NaN,Oomycota,Saprolegniales,Saprolegniaceae,Aphanomyces,Aphanomyces euteiches,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,4.831190,0.000000


### Pathway Metagenomics

Each column is a skin biopsy sample from a crew member pre- or post- flight.

Each row is a different molecular pathway whose activity was quantified within that sample.

In [39]:
pathway_metagenomics_skin = pd.read_csv(
    'https://osdr.nasa.gov/geode-py/ws/studies/OSD-574/download?source=datamanager'
    '&file=GLDS-566_GMetagenomics_Pathway-abundances-cpm_GLmetagenomics.tsv',
    index_col=0, sep='\t')
pathway_metagenomics_skin = add_col_metadata(pathway_metagenomics_skin)
pathway_metagenomics_skin.head()

,C001_L-44_DEL_Abundance-CPM,C001_R+1_DEL_Abundance-CPM,C002_L-44_DEL_Abundance-CPM,C002_R+1_DEL_Abundance-CPM,C003_L-44_DEL_Abundance-CPM,C003_R+1_DEL_Abundance-CPM,C004_L-44_DEL_Abundance-CPM,C004_R+1_DEL_Abundance-CPM
# Pathway,,,,,,,,
UNMAPPED,104601.00000,185185.0,525379.000,570481.0,147540.0000,140201.0000,244947.00000,342548.0000
UNINTEGRATED,857097.00000,778117.0,452089.000,396336.0,819130.0000,824883.0000,715281.00000,623923.0000
"12DICHLORETHDEG-PWY: 1,2-dichloroethane degradation",0.00000,0.0,0.000,0.0,0.0000,0.0000,8.61217,0.0000
1CMET2-PWY: folate transformations III (E. coli),22.97510,0.0,102.238,0.0,15.6024,27.9638,65.88670,75.4531
ALLANTOINDEG-PWY: superpathway of allantoin degradation in yeast,1.39909,0.0,0.000,0.0,0.0000,0.0000,0.00000,0.0000


### Gene Family Metagenomics

Each column is a skin biopsy sample from a crew member pre- or post- flight.

Each row is a different gene family whose activity was quantified within that sample.

In [40]:
genefam_metagenomics_skin = pd.read_csv(
    'https://osdr.nasa.gov/geode-py/ws/studies/OSD-574/download?source=datamanager'
    '&file=GLDS-566_GMetagenomics_Gene-families-cpm_GLmetagenomics.tsv',
    index_col=0, sep='\t')
genefam_metagenomics_skin = add_col_metadata(genefam_metagenomics_skin)
genefam_metagenomics_skin.head()

,C001_L-44_DEL_Abundance-CPM,C001_R+1_DEL_Abundance-CPM,C002_L-44_DEL_Abundance-CPM,C002_R+1_DEL_Abundance-CPM,C003_L-44_DEL_Abundance-CPM,C003_R+1_DEL_Abundance-CPM,C004_L-44_DEL_Abundance-CPM,C004_R+1_DEL_Abundance-CPM
# Gene Family,,,,,,,,
UNMAPPED,104601.0,185185.0000,525379.0,570481.0,147540.000000,140201.0,244947.0000,342548.00000
UniRef90_A0A009MJC6,0.0,0.0000,0.0,0.0,0.582863,0.0,0.0000,0.00000
UniRef90_A0A009Y5I8,0.0,0.0000,0.0,0.0,0.000000,0.0,0.0000,1.59667
UniRef90_A0A010NNI4,0.0,10.0082,0.0,0.0,0.000000,0.0,0.0000,0.00000
UniRef90_A0A010NUY0,0.0,0.0000,0.0,0.0,0.000000,0.0,1.6162,0.00000


# Data Export

In [41]:
import os
os.makedirs("data/processed", exist_ok=True)

exports = {
    # ── Direct analyte panels ──────────────────────────────────────────────
    "cmp_metabolic_panel":          metabolic_blood,
    "cbc_complete_blood_count":     cbc,
    "urine_inflammation_panel":     urine,
    "immune_cytokines_eve":         immune_eve,
    "immune_cytokines_alamar":      immune_alamar,
    "cardiac_cytokines_eve":        cardio_eve,
    # ── Omics panels ───────────────────────────────────────────────────────
    "plasma_metabolomics":          metabolomics,
    "evp_proteomics":               evp,
    "plasma_proteomics":            protein,
    "whole_blood_rnaseq":           rna_blood,
    "whole_blood_m6A":              m6A,
    "pbmc_snrnaseq":                snrnaseq,
    "pbmc_snATACseq":               snatacseq,
    "pbmc_vdj":                     vdj,
    "skin_spatial_transcriptomics": spatial,
    # ── Metagenomics ───────────────────────────────────────────────────────
    "crew_swab_taxonomy":           tax_metagenomics_crew,
    "stool_taxonomy":               tax_metagenomics_stool,
    "skin_taxonomy":                tax_metagenomics_skin,
    "dragon_capsule_taxonomy":      tax_metagenomics_dragon,
}

for name, df in exports.items():
    path = f"data/processed/{name}.csv"
    df.to_csv(path)
    # For wide tables, also save the column metadata sidecar
    if hasattr(df, "attrs") and "meta" in df.attrs:
        df.attrs["meta"].to_csv(f"data/processed/{name}__meta.csv")
    print(f"✅ {name} → {path}  ({len(df)} rows × {len(df.columns)} cols)")

print("\nAll 19 datasets written to data/processed/")


✅ cmp_metabolic_panel → data/processed/cmp_metabolic_panel.csv  (57 rows × 31 cols)
✅ cbc_complete_blood_count → data/processed/cbc_complete_blood_count.csv  (553 rows × 11 cols)
✅ urine_inflammation_panel → data/processed/urine_inflammation_panel.csv  (22 rows × 409 cols)
✅ immune_cytokines_eve → data/processed/immune_cytokines_eve.csv  (142 rows × 31 cols)
✅ immune_cytokines_alamar → data/processed/immune_cytokines_alamar.csv  (406 rows × 30 cols)
✅ cardiac_cytokines_eve → data/processed/cardiac_cytokines_eve.csv  (18 rows × 31 cols)
✅ plasma_metabolomics → data/processed/plasma_metabolomics.csv  (656 rows × 6 cols)
✅ evp_proteomics → data/processed/evp_proteomics.csv  (527 rows × 6 cols)
✅ plasma_proteomics → data/processed/plasma_proteomics.csv  (1765 rows × 6 cols)
✅ whole_blood_rnaseq → data/processed/whole_blood_rnaseq.csv  (61852 rows × 38 cols)
✅ whole_blood_m6A → data/processed/whole_blood_m6A.csv  (57318 rows × 22 cols)
✅ pbmc_snrnaseq → data/processed/pbmc_snrnaseq.csv  (82